# Lecture 4: exercise solutions

Worked solutions to the exercises in Lecture 4. The complete code is in `solution_files/`:
`to_debug_solution.py` and its regression test `test_to_debug_solution.py`, `mfe_solution.py`,
`fourier_series_solution.py` (the live debugging demo, fixed), and
`l4mod-complete/` (the fixed module with doctests, its tests, and the Sphinx `docs` folder).

## Exercise: generating exceptions

One snippet per exception (each cell below raises it on purpose):

#### 1. `ImportError`: the module exists, but not the name

In [1]:
from math import sin_and_cos

ImportError: cannot import name 'sin_and_cos' from 'math' (/home/rhodri/miniconda3/envs/mpm2026/lib/python3.14/lib-dynload/math.cpython-314-x86_64-linux-gnu.so)

#### 2. `ModuleNotFoundError`: no such module

In [2]:
import numpyy

ModuleNotFoundError: No module named 'numpyy'

#### 3. `ZeroDivisionError`

In [3]:
1 / 0

ZeroDivisionError: division by zero

#### 4. `IndexError`: a list of 3 elements has indices 0 to 2

In [4]:
[1, 2, 3][3]

IndexError: list index out of range

#### 5. `TypeError`: an operation on incompatible types

In [5]:
"2" + 2

TypeError: can only concatenate str (not "int") to str

#### 6. `SyntaxError`: not valid Python (in Python 3, `print` is a function)

In [6]:
print "hello"

SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)? (2253489653.py, line 1)

#### 7. `IndentationError`

In [7]:
def f():
return 1

IndentationError: expected an indented block after function definition on line 1 (341325504.py, line 2)

#### 8. `RecursionError`: a function that calls itself without stopping

In [8]:
def f(n):
    return f(n + 1)


f(0)

RecursionError: maximum recursion depth exceeded

Two things worth noticing. `ModuleNotFoundError` is a subclass of `ImportError`, so
`except ImportError:` catches both. And `SyntaxError` and `IndentationError` are raised when Python
*parses* the code, before any of it runs: that is why their tracebacks point at the source text
rather than at a function call, and why a script with a syntax error doesn't run at all.

## Exercise: static analysis

`flake8 .`, run in the `files` folder, reports 40 messages. Only three are `F` codes, which point to code that
is wrong or unnecessary:

```
./kiss.py:13:1: F811 redefinition of unused 'add_numbers' from line 2
./l4mod/l4mod.py:1:1: F401 'numpy as np' imported but unused
./wiener_filtering.py:4:1: F401 'scipy as sp' imported but unused
```

- `F401`: a module is imported but never used (`wiener_filtering.py` and `l4mod.py`).
- `F811`: `kiss.py` defines `add_numbers` twice, so the first definition is silently replaced. Here
  that is deliberate, to show both versions, but in real code it is usually a bug.

The rest are `pycodestyle` style messages (`E` and `W`): lines over 79 characters (`E501`), missing
blank lines between functions (`E302`, `E305`), spacing around operators and commas (`E225`, `E231`),
and trailing whitespace (`W291`, `W293`). None changes what the code does, but a consistent style
makes code easier to read, and editors can fix most of these automatically.

## Demo: the Fourier series (`fourier_series.py`)

The partial sums plotted by `files/fourier_series.py` should approach $x^2$ as $N$ grows, but they
don't. The bug is in `eval_fc`: each term is divided by `N**2`, the number of terms, instead of
`i**2`, the square of its own index $n$:

```python
f += 4*(-1)**i/N**2*np.cos(i*x)   # bug
f += 4*(-1)**i/i**2*np.cos(i*x)   # fix
```

A print statement in the loop (or `p 4*(-1)**i/N**2` at a `breakpoint()`) shows it: with $N = 100$
every term has a coefficient of size $4/100^2 = 0.0004$, where the first should be $4$. With the fix,
the sum with $N = 100$ is within 0.04 of $x^2$ everywhere on $[-\pi, \pi]$; with the bug it is up to
6.5 away. `solution_files/fourier_series_solution.py` is the fixed script.

## Exercise: debugging with `pdb` (`to_debug.py`)

**Bug 1: the crash.** Running the script ends with

```
TypeError: unsupported operand type(s) for +: 'int' and 'tuple'
```

in `bench_optimizer`'s `sum`. In the debugger (`python -m pdb to_debug.py`, `c` to run to the
error), `p apply_optimizer(optimizer, func, a, b)` shows a one-element tuple such as `(5,)`, not a number. The cause is
the stray comma at the end of `apply_optimizer`'s `return` line. Remove it.

**Bug 2: the quiet one.** The script now runs, but prints something like

```
              brenth:   601123 total function calls
              brentq:        0 total function calls
              ridder:        0 total function calls
              bisect:        0 total function calls
```

`itertools.product` returns an *iterator*, which can be used only once: the first call of
`bench_optimizer` uses it up, so later optimizers get an empty grid. Build a list instead:
`param_grid = list(product(FUNCTIONS, random_a, random_b))`.

**The regression test** (`solution_files/test_to_debug_solution.py`) captures what
`compare_optimizers` prints with `pytest`'s `capsys` fixture and checks that every optimizer reports
a positive count. It fails with bug 2 and passes once it is fixed.

## Exercise: MFE (`mfe.py`)

`load_columns` keeps `y` as a column (`table[:, 1:]`, shape `(5, 1)`), but `y_fit = m*x + c` has
shape `(5,)`. In `rms_error`, `y - y_fit` therefore broadcasts to a `(5, 5)` array of every measured
value minus every fitted value. No error is raised, but the RMS error is 3.983 instead of 0.146. The
minimal failing example needs no data and no fit:

```python
import numpy as np

y = np.array([[1.0], [2.0], [3.0]])   # a column, shape (3, 1)
y_fit = np.array([1.0, 2.0, 3.0])     # shape (3,)
print((y - y_fit).shape)              # (3, 3), not (3,)
```

Fix: take a 1D column, `y = table[:, 1]`. To stop the bug coming back silently, have `rms_error`
refuse mismatched shapes, and test that it does:

```python
def rms_error(y, y_fit):
    if np.shape(y) != np.shape(y_fit):
        raise ValueError(f"shapes differ: {np.shape(y)} and {np.shape(y_fit)}")
    return np.sqrt(np.mean((y - y_fit)**2))


def test_rms_error_of_an_exact_fit_is_zero():
    y = np.array([1.0, 2.0, 3.0])
    assert rms_error(y, y) == 0.0


def test_rms_error_rejects_mismatched_shapes():
    with pytest.raises(ValueError):
        rms_error(np.ones((3, 1)), np.ones(3))
```

## Exercises: test classes, `doctest`, coverage and Sphinx (`l4mod`)

`solution_files/l4mod-complete/test_l4mod.py` has a `TestUnits` class (parametrised tests of
`matmul` and `zeromat`) and a `TestModule` class (`gauss`), plus a `pytest.raises` test for
`matmul`'s incompatible-dimensions error.

**The bug the tests find.** `gauss`'s docstring says it accepts NumPy arrays, and it does for lists,
but with arrays and a matrix whose rows need swapping it returns a determinant of 0 and NaNs. Run
against the original `l4mod.py`, the tests give:

```
FAILED test_l4mod.py::TestModule::test_gauss_accepts_numpy_arrays - assert np...
1 failed, 13 passed, 2 warnings
```

`pytest --pdb` stops in the failing test. Stepping through `gauss` shows two rows becoming identical
at the row swap `a[i], a[k] = a[k], a[i]`. For lists that line swaps two row objects. For a NumPy
array, `a[k]` is a *view* of row `k` (Lecture 2), so after `a[i] = a[k]` has copied row `k` into row
`i`, the "old" row `i` that should go into row `k` already holds row `k`'s values. Both rows end up
the same, and the matrix becomes singular. The fix in the solution copies the inputs into lists of
lists at the start (which also replaces the `copy.deepcopy` calls); `a[[i, k]] = a[[k, i]]` would
fix the swap for arrays only. The test stays in the suite as a regression test.

**Doctests.** Each function's docstring now has an `Examples` section, e.g.

```
>>> matmul([[1, 2], [3, 4]], [[5], [6]])
[[17], [39]]
```

`pytest --doctest-modules` runs them along with the tests: 17 passed.

**Coverage** (`pytest --doctest-modules --cov=l4mod --cov-report=term-missing`):

```
Name       Stmts   Miss  Cover   Missing
----------------------------------------
l4mod.py      56     12    79%   124-135
----------------------------------------
TOTAL         56     12    79%
```

The only lines not run are the `if __name__ == "__main__":` block at the end, which the tests never
execute. That is fine here; if it mattered, a test could run the module as a script.

**Sphinx.** With `docs/conf.py` and `docs/index.rst` as in `solution_files/l4mod-complete/docs/`,
`sphinx-build -b html docs docs/html` builds the HTML documentation from the docstrings, and with
LaTeX installed, `sphinx-build -b latex docs docs/tex` followed by `pdflatex l4mod.tex` (twice) in
`docs/tex` builds the PDF.